## 00 — What Are Vector Tiles?

In Module 06 we identified four pain points in our handbuilt system:
1. Slow startup (load + index all data upfront)
2. Verbose GeoJSON format
3. Whole file always resident in memory
4. No streaming — unused regions still loaded

**Vector tiles** are the data format that solves all four. This notebook explains what they are before we use the tool that generates them.

## The Core Idea — Pre-Sliced, Pre-Indexed Data

Instead of four large files that cover the whole world, a vector tile pyramid pre-slices the world into thousands of small tiles — one per `{zoom}/{x}/{y}` address — before the user ever opens the map.

```
Zoom 0: 1 tile  (whole world)
Zoom 1: 4 tiles (quadrants)
Zoom 2: 16 tiles
...
Zoom 14: 268 million tiles (most empty)
```

When the user views a map, only the tiles that are currently visible are fetched. A user in Paris at zoom 12 receives ~12 tiles covering roughly 5km × 5km each. Siberia is never touched.

## How Each Tile Maps to Our System

Every choice we made manually now happens automatically inside the tile generator:

| What we built | What the tile system does |
|---------------|---------------------------|
| 4 LOD files at fixed epsilons | Per-zoom simplification baked into each tile |
| Grid index bucketing features into cells | Each tile IS a cell — features are pre-bucketed by definition |
| Viewport bbox culling | Each tile covers a fixed bbox — fetching only nearby tiles IS the cull |
| Zoom decision function | The tile URL scheme `/{z}/{x}/{y}` carries the zoom level |
| GeoJSON text format | MVT binary encoding — coordinates as integers, ~5× smaller |
| Whole file loaded at startup | Each tile fetched on demand, ~50–200 KB each |

## The Tile Coordinate System

Tiles use `(z, x, y)` addressing. At zoom `z`, the world is divided into a `2^z × 2^z` grid.

Given a longitude/latitude, we can compute its tile address:

In [1]:
import math

def lon_lat_to_tile(lon, lat, zoom):
    """Return the (z, x, y) tile address for a geographic point at a given zoom."""
    n = 2 ** zoom
    x = int((lon + 180) / 360 * n)
    lat_rad = math.radians(lat)
    y = int((1 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2 * n)
    return zoom, x, y

# Paris
for zoom in [2, 5, 8, 12]:
    z, x, y = lon_lat_to_tile(2.35, 48.86, zoom)
    print(f"  zoom {z:>2}  tile ({z}/{x}/{y})")

  zoom  2  tile (2/2/1)
  zoom  5  tile (5/16/11)
  zoom  8  tile (8/129/88)
  zoom 12  tile (12/2074/1409)


## The MVT Binary Format

Mapbox Vector Tiles (MVT) store geometry as integers instead of floating-point text.

Each tile has a local coordinate space of 4096 × 4096 units. A coordinate like `(48.8566, 2.3522)` is projected into this space and stored as two small integers (e.g., `(2047, 1803)`).

This gives:
- **5–10× smaller files** vs. GeoJSON (integers compress better than decimal strings)
- **Faster parse** (no string-to-float conversion)
- **Lossy but controlled precision** (4096 units per tile at zoom 14 ≈ 2m resolution)

## The PMTiles Format

Traditionally, tile pyramids were stored in SQLite databases (`.mbtiles`) or as millions of individual files on a server.

**PMTiles** is a newer single-file format that stores the entire tile pyramid in one `.pmtiles` file, arranged so that spatially nearby tiles are stored close together on disk. A client can fetch just the tiles it needs using HTTP range requests — no tile server required, just a static file on any CDN.

For our purposes: `tippecanoe` can output either `.mbtiles` or `.pmtiles`.

## Exercise A

At zoom 12, the world is divided into `2^12 × 2^12 = 4096 × 4096 = ~16.7 million` tiles.

1. How many tiles cover Western Europe at zoom 12? (Approximate using the bounding box [-10, 35, 30, 60])
2. If each tile is 100 KB on average, how much data would the user need to download to view all of Western Europe at zoom 12?

Compare that to loading our `extra_fine` GeoJSON for the same region.

In [2]:
import math

# Western Europe bounding box
west, south, east, north = -10, 35, 30, 60

# Zoom level
z = 12
n = 2 ** z   # number of tiles in each direction (4096)

# Convert lon/lat to tile coordinates
def lon_to_xtile(lon, zoom):
    return (lon + 180.0) / 360.0 * (2 ** zoom)

def lat_to_ytile(lat, zoom):
    lat_rad = math.radians(lat)
    return (1 - math.log(math.tan(lat_rad) + 1 / math.cos(lat_rad)) / math.pi) / 2 * (2 ** zoom)

# Compute tile ranges
x_min = lon_to_xtile(west, z)
x_max = lon_to_xtile(east, z)
y_min = lat_to_ytile(north, z)
y_max = lat_to_ytile(south, z)

# Number of tiles covering Western Europe
tile_count = int((x_max - x_min) * (y_max - y_min))

print("Approx tile count for Western Europe at zoom 12:", tile_count)

# Download size at 100 KB per tile
download_mb = tile_count * 100_000 / 1_000_000
print(f"Estimated download size: {download_mb:.1f} MB")

# Compare to extra_fine GeoJSON size
import os
geojson_size_mb = os.path.getsize("../../data/lod/railroads_extra_fine.geojson") / 1_000_000
print(f"extra_fine GeoJSON size: {geojson_size_mb:.1f} MB")



Approx tile count for Western Europe at zoom 12: 197035
Estimated download size: 19703.5 MB
extra_fine GeoJSON size: 19.0 MB


## Exercise B

The tile coordinate formula uses the Web Mercator projection — the same projection used by Google Maps, OpenStreetMap, and virtually all web maps.

Web Mercator distorts areas significantly near the poles. Greenland appears roughly the same size as Africa on a Web Mercator map, even though Africa is ~14× larger.

Does this distortion affect the **accuracy** of our railroad visualization? Explain why or why not in 3–4 sentences.

Web Mercator’s distortion does not harm the accuracy of our railroad visualization because we never use the projection to measure real‑world distances or areas. The distortion only affects how shapes look on the screen, not the underlying geometry or the spatial queries, which all operate in latitude/longitude space. Our grid index, bounding‑box tests, and LOD selection all run in geographic coordinates, so the projection is just a display layer. In short, Web Mercator may stretch Greenland, but it does not change which railroad features appear or how accurately they are selected.

## Check Your Understanding

The tile grid at zoom 14 has ~268 million possible tile addresses. Most tiles — over oceans, deserts, and polar regions — contain no data.

Both `.mbtiles` (SQLite) and `.pmtiles` (single file) only store non-empty tiles. Why is this critical, and how does it relate to the `scalerank` filtering decision we made in our LOD pipeline?

---

**Why storing only non-empty tiles is critical**
At zoom 14 there are 268 million possible tiles, but only a tiny fraction contain any railroad geometry. If a tile format stored every tile-including empty ocean, desert, and polar tiles-the dataset would explode to hundreds of gigabytes and be completely unusable on real networks. By storing only non-empty tiles, formats like mbtiles and pmtiles keep the dataset compact, fast to transfer, and cheap to cache on CDNs.

**How this relates to the scalerank filtering decision**
Our scalerank filtering plays the same role: it prevents us from generating "empty" or "low-value" content at coarse zoom levels. Without scalerank filtering, many coarse-zoom tiles would contain tiny, low-importance rail segments that don't matter visually but would still force the system to store and serve far more tiles. In other words, both systems avoid wasting storage and bandwidth on data that provides no value at a given scale, which is essential for performance, especially at high zoom levels where tile counts explode.


## Next

In [01 — Using Tippecanoe](./01-Using_Tippecanoe.ipynb), we run `tippecanoe` on the raw railroad GeoJSON and map each of its flags to decisions we already made.